# Module 02 — Lazy Execution

Transformations build a DAG; actions execute it. We'll use **expensive** pipelines so duplicated work is obvious in the UI and in wall-clock time.

In [13]:
import sys
import time
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from pyspark.sql import Window
from pyspark.sql.functions import col, row_number, sum as _sum, avg, count
from pyspark import StorageLevel
from pyspark.sql import Observation
from shared.spark_session import get_spark, ui_links
from shared import config

spark = get_spark('module-02-lazy-execution')
ui_links()

{'master_ui': 'http://localhost:8080',
 'app_ui': 'http://localhost:4040',
 'worker_1_ui': 'http://localhost:8081',
 'worker_2_ui': 'http://localhost:8082',
 'app_ui_internal': 'http://jupyter-lab:4040'}

In [2]:
trips = spark.read.parquet(*config.yellow_taxi_paths())

## Expensive Pipeline Helper

Filter → window → aggregate → sort. Each action on this hurts.

In [3]:
def expensive_pipeline(df):
    w = Window.partitionBy('PULocationID').orderBy(col('total_amount').desc())
    return (
        df.filter('total_amount > 0 AND trip_distance > 0')
        .withColumn('rn', row_number().over(w))
        .filter('rn <= 100')
        .groupBy('VendorID', 'PULocationID')
        .agg(_sum('total_amount').alias('total'), avg('trip_distance').alias('avg_dist'))
        .orderBy(col('total').desc())
    )

## 1. Lazy Transformations

Build the pipeline — no jobs until an action.

In [4]:
pipeline = expensive_pipeline(trips)
print('Pipeline defined. Check Jobs tab — no new job yet.')

Pipeline defined. Check Jobs tab — no new job yet.


In [5]:
t0 = time.perf_counter()
n = pipeline.count()
print(f'Rows: {n:,}  Time: {time.perf_counter() - t0:.1f}s')

[Stage 1:===========================================>              (9 + 3) / 12]

Rows: 508  Time: 3.0s


## 2. Branching DAG — Duplicated Work

The same expensive ancestor is reused in **two actions**. Without caching, Spark re-runs the entire pipeline twice — watch duplicated stages in the UI.

In [6]:
base = expensive_pipeline(trips)

t0 = time.perf_counter()
a = base.count()
t1 = time.perf_counter()
b = base.filter('total > 100').count()
t2 = time.perf_counter()

print(f'Action 1 (count):       {a:,} rows  {t1 - t0:.1f}s')
print(f'Action 2 (filter+count): {b:,} rows  {t2 - t1:.1f}s')
print(f'Total: {t2 - t0:.1f}s  — compare to a single action above')

[Stage 13:============================>                            (6 + 6) / 12]

Action 1 (count):       508 rows  0.9s
Action 2 (filter+count): 491 rows  1.0s
Total: 1.9s  — compare to a single action above


## 3. Three Actions — Even Worse

A third branch multiplies the pain. Note shuffle stages repeating in the UI.

In [7]:
base = expensive_pipeline(trips)

t0 = time.perf_counter()
r1 = base.count()
r2 = base.filter('avg_dist > 2').count()
r3 = base.groupBy('VendorID').count().count()
print(f'Three actions total: {time.perf_counter() - t0:.1f}s')

[Stage 31:===================================================>    (11 + 1) / 12]

Three actions total: 2.8s


## 4. Mitigation: Cache

Materialize once, reuse everywhere.

In [8]:
base_cached = expensive_pipeline(trips).persist(StorageLevel.MEMORY_AND_DISK)

t0 = time.perf_counter()
base_cached.count()  # materializes
t_mat = time.perf_counter()
a = base_cached.count()
b = base_cached.filter('total > 100').count()
c = base_cached.groupBy('VendorID').count().count()
t_end = time.perf_counter()

print(f'Materialize: {t_mat - t0:.1f}s')
print(f'3 cached actions: {t_end - t_mat:.1f}s')
base_cached.unpersist()

Materialize: 2.0s
3 cached actions: 1.0s


DataFrame[VendorID: int, PULocationID: int, total: double, avg_dist: double]

## 5. Mitigation: Checkpoint

Breaks lineage by writing intermediate results to storage.

In [9]:
spark.sparkContext.setCheckpointDir(str(config.CHECKPOINT_DIR))

cp = expensive_pipeline(trips).checkpoint()
t0 = time.perf_counter()
cp.count()
print(f'Checkpoint materialize: {time.perf_counter() - t0:.1f}s')

t0 = time.perf_counter()
cp.count()
cp.filter('total > 50').count()
print(f'Post-checkpoint actions: {time.perf_counter() - t0:.1f}s')

26/10/11 19:45:28 WARN SparkContext: Spark is not running in local mode, therefore the checkpoint directory must not be on the local filesystem. Directory '/workspace/data/output/checkpoints' appears to be on the local filesystem.
                                                                                

Checkpoint materialize: 0.1s
Post-checkpoint actions: 0.2s


## 6. Spark Observation — Metrics in a Single Action

In [14]:
obs = Observation('metrics')
observed = expensive_pipeline(trips).observe(obs, count('*'))
observed.count()
print(obs.get)

{'count(1)': 508}


## 7. Live Discussion (Walk Through Together)

1. How many shuffle stages repeated across the three uncached actions?
2. How much time did caching save?
3. When would you choose checkpoint over cache?

## Key Takeaways

1. Only **actions** trigger execution.
2. Shared ancestors in branched DAGs cause **duplicate work** — expensive pipelines make this visible.
3. Use `.cache()`, `.checkpoint()`, and `Observation` to control cost.